## Created by:
- Mikołaj Nowak 151813
- Igor Szymczak 160280

## Task
Create a search engine that based on a query will return closest matching texts (you can use titles of wikipedia articles you have downloaded). Can you extend the edit distance method to:
  

*   penalize less errors if two letters are close to each other on a keyboard e.g. m-n
*   prioritize matching from the begining of a word. If a query is "to" then "Tomahawk" is a better match than "Potato" since we assume that's in line with users' behaviour

Propose and implement two additional extensions, show on some examples how does your approach work. As always 144h from end of this class to send the project

## 1. Data reading

In [1]:
titles_file = "article_titles.txt"

with open(titles_file, "r", encoding="utf-8") as f:
    article_titles = [line.strip() for line in f.readlines() if line.strip()]

## 2. Calculate penalties for typos

In [2]:
keyboard_rows = [
    "qwertyuiop",
    "asdfghjkl",
    "zxcvbnm"
]

key_positions = {}
for row_idx, row in enumerate(keyboard_rows):
    for col_idx, char in enumerate(row):
        key_positions[char] = (row_idx, col_idx)


def keyboard_distance(c1, c2):
    """
    Returns Manhattan distance between two lowercase keyboard characters.
    If a character is not on the keyboard (e.g. punctuation), return a large distance.
    """
    c1 = c1.lower()
    c2 = c2.lower()

    if c1 not in key_positions or c2 not in key_positions:
        return 10  # treat unknown characters as far away

    r1, c1_pos = key_positions[c1]
    r2, c2_pos = key_positions[c2]

    return abs(r1 - r2) + abs(c1_pos - c2_pos)


def substitution_penalty(c1, c2):
    """
    Computes the substitution cost based on keyboard proximity.
    Distance = 1 → penalty 0.5
    Distance = 2 → penalty 0.75
    Distance >= 3 → penalty 1.0
    """
    dist = keyboard_distance(c1, c2)

    if dist == 1:
        return 0.5
    elif dist == 2:
        return 0.75
    else:
        return 1.0

In [3]:
# Quick tests:
test_pairs = [("m", "n"), ("i", "o"), ("t", "g"), ("a", "p")]

for a, b in test_pairs:
    print(f"{a} → {b}: dist={keyboard_distance(a,b)}, penalty={substitution_penalty(a,b)}")

m → n: dist=1, penalty=0.5
i → o: dist=1, penalty=0.5
t → g: dist=1, penalty=0.5
a → p: dist=10, penalty=1.0


## 3. Improved Levenshtein Distance Scoring

Standard Levenshtein distance works well for short strings, but it produces misleading results when comparing a short query against long article titles. Even if a long title contains a perfect substring match, the full-string edit distance becomes very large simply because the title has many extra words. This causes unrelated short strings to appear “closer” than relevant long ones.

To fix this, we introduce a **chunk-based Levenshtein scoring method** with keyboard-aware substitution penalties. The improvements are:

### **3.1. Chunking based on the query structure**
Instead of comparing the full query to the entire title, we:
- count how many spaces are in the query (e.g., `"atrificial life"` → 1 space → 2-word chunk)
- slide a window of that size over each article title
- compute the Levenshtein distance only on those smaller chunks  
This ensures long titles are not unfairly penalized.

### **3.2. Keyboard-aware substitution penalties**
When characters differ, the penalty depends on how close they are on a physical keyboard:
- distance 1 → penalty 0.5  
- distance 2 → penalty 0.75  
- otherwise → penalty 1  
This makes typical typing mistakes cost less than random substitutions.

### **3.3. Positional bias**
If a chunk occurs at the **beginning or end** of the title, we slightly reduce the distance. This promotes matches that align naturally with title boundaries.

### **3.4. Normalized similarity score**
After computing the chunk distance, we convert it into a 0–1 similarity score:

$$
\text{score} = 1 - \frac{\text{distance}}{\text{len(query)}}
$$

This normalizes scores so that:
- perfect matches score near 1  
- mismatches approach 0  
- scores remain comparable regardless of title length


In [4]:
def levenshtein_base(s1, s2):

    if len(s1) < len(s2):
        return levenshtein_base(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = list(range(len(s2) + 1))

    for i, c1 in enumerate(s1):
        current_row = [i + 1]

        for j, c2 in enumerate(s2):

            insertions = previous_row[j + 1] + 1
            deletions  = current_row[j] + 1

            if c1 == c2:
                sub_cost = 0
            else:
                sub_cost = substitution_penalty(c1, c2)

            substitutions = previous_row[j] + sub_cost

            current_row.append(min(insertions, deletions, substitutions))

        previous_row = current_row

    return previous_row[-1]


In [5]:
import math
import re

def chunk_based_similarity(query, title):
    query = re.sub(r'\s+', ' ', query.strip().lower()).strip()
    title = title.strip().lower()

    space_count = query.count(" ")
    chunk_sizes = [space_count, max(space_count - 1, 1), space_count + 1]

    words = title.split()
    best_score = -9999

    for chunk_size in chunk_sizes:
        if len(words) < chunk_size:
            dist = levenshtein_base(query, title)
            score = 1 - dist / max(len(query), 1)
            best_score = max(best_score, score)
            continue

        for i in range(len(words) - chunk_size + 1):
            chunk = " ".join(words[i:i + chunk_size]).strip()
            dist = levenshtein_base(query, chunk)
            if i == 0 or i + chunk_size == len(words):
                dist -= 1
            score = 1 - dist / max(len(query), 1)
            best_score = max(best_score, score)

    return best_score

## 4. Strengths

The improved Levenshtein scoring handles both **typographical errors** and **long titles** effectively. By using keyboard-aware substitution penalties, common typos such as `"intelligemce"` instead of `"intelligence"` are recognized with a smaller cost. The chunk-based comparison prevents long titles from being unfairly penalized, allowing relevant matches to be ranked highly even when they contain multiple words. Consecutive spaces in queries are also normalized, so extra or missing spaces do not affect the similarity scoring. Overall, this method produces accurate and robust similarity scores for real-world search queries.

In [6]:
query = "   atrificial    intelligemce   "
scores = []

for title in article_titles:
    score = chunk_based_similarity(query, title)
    scores.append((title, score))

scores = sorted(scores, key=lambda x: x[1], reverse=True)
scores[:5]


[('A.I. Artificial Intelligence', 0.9782608695652174),
 ('Association for the Advancement of Artificial Intelligence',
  0.9782608695652174),
 ('Artificial intelligence in Wikimedia projects', 0.9782608695652174),
 ('Artificial intelligence visual art', 0.9782608695652174),
 ('Applications of artificial intelligence', 0.9782608695652174)]

## 5. Weaknesses

The current chunk-based Levenshtein approach handles typos and long titles well, but it struggles with queries that contain **midword spaces** or words in **reversed order**. For example, a query like `"intell igemce atrifi cial"` splits words in unusual ways, breaking some of the chunks. Similarly, if query words appear in a different order than in the title, the fixed consecutive-word chunking cannot fully capture the match.  

As a result, relevant articles are still returned (e.g., articles about "Artificial Intelligence"), but the **similarity scores are lower** than for perfectly aligned queries.

In [8]:
query = "intell igemce atrifi cial"
scores = []

for title in article_titles:
    score = chunk_based_similarity(query, title)
    scores.append((title, score))

scores = sorted(scores, key=lambda x: x[1], reverse=True)
scores[:5]

[('Artificial intelligence visual art', 0.6699999999999999),
 ('Artificial intelligence arms race', 0.6699999999999999),
 ('List of artificial intelligence projects', 0.6699999999999999),
 ('Artificial intelligence in fiction', 0.6599999999999999),
 ('Intelligence amplification', 0.6599999999999999)]